# 03 — HOG + SVM

HOG mô tả hướng gradient trên lưới không gian cố định. Ảnh RGB224 được đổi sang xám; 14×14 cell và 13×13 block tạo `13×13×2×2×9 = 6.084` chiều.


In [ ]:
# KAGGLE_INPUT_SETUP: tìm code và DATA trong các Dataset đã gắn.
from pathlib import Path
import os

kaggle_input = Path('/kaggle/input')
dataset_roots = []
if kaggle_input.is_dir():
    dataset_roots.extend(kaggle_input.glob('datasets/*/*'))
    dataset_roots.extend(kaggle_input.glob('*'))
dataset_roots = [p for p in dataset_roots if p.is_dir()]

project_hint = os.environ.get('CS231_PROJECT_ROOT')
project_candidates = ([Path(project_hint)] if project_hint else
                      [Path.cwd(), *Path.cwd().parents, *dataset_roots])
project_matches = sorted({p.resolve() for p in project_candidates
                          if (p / 'src/SVM/pipeline.py').is_file()})
if len(project_matches) != 1:
    raise FileNotFoundError(
        'Cần gắn đúng Dataset code chứa src/SVM/pipeline.py; '
        'nếu có nhiều bản, đặt CS231_PROJECT_ROOT. '
        f'Tìm thấy: {project_matches}'
    )
project = project_matches[0]

data_hint = os.environ.get('CS231_DATA_ROOT')
data_candidates = ([Path(data_hint)] if data_hint else
                   [project / 'DATA', *(p / 'DATA' for p in dataset_roots)])
data_matches = sorted({p.resolve() for p in data_candidates
                       if (p / 'train/10000').is_dir()})
if len(data_matches) != 1:
    raise FileNotFoundError(
        'Cần gắn đúng Dataset DATA chứa DATA/train/10000; '
        'nếu có nhiều bản, đặt CS231_DATA_ROOT. '
        f'Tìm thấy: {data_matches}'
    )
data = data_matches[0]

os.environ['CS231_PROJECT_ROOT'] = str(project)
os.environ['CS231_DATA_ROOT'] = str(data)
print('Code:', project, (project / 'src/SVM/pipeline.py').is_file())
print('DATA:', data, (data / 'train/10000').is_dir())


In [ ]:
PREVIOUS_ARTIFACT = 'hsv12/svm_grid_dev.csv'
# PREVIOUS_NOTEBOOK_OUTPUT: khôi phục toàn bộ kết quả của notebook ngay trước đó.
from pathlib import Path
from shutil import copytree
import os

if Path('/kaggle/working').is_dir():
    notebook_root = Path('/kaggle/input/notebooks')
    attached = sorted(p for p in notebook_root.glob('*/*') if p.is_dir())
    if len(attached) != 1:
        raise FileNotFoundError(
            'Hãy gắn đúng một Notebook Output của bước ngay trước đó. '
            f'Đang thấy: {attached}'
        )
    configs = [p for p in attached[0].rglob('experiment_config.json')
               if p.parent.name == 'inputs' and p.parent.parent.name == 'results_224']
    if len(configs) != 1:
        raise FileNotFoundError(
            'Notebook Output đã gắn phải chứa results_224/inputs/experiment_config.json. '
            f'Tìm thấy: {configs}'
        )
    source = configs[0].parents[1]
    expected = source / PREVIOUS_ARTIFACT
    if not expected.is_file():
        raise FileNotFoundError(
            f'Output không phải bản chạy thành công của bước ngay trước: thiếu {expected}'
        )
    target = Path(os.environ.get('CS231_SVM_RESULTS_ROOT',
                                 '/kaggle/working/CS231_SVM/results_224'))
    copytree(source, target, dirs_exist_ok=True)
    for relative in ('inputs/train_manifest.csv', 'inputs/dev_manifest.csv'):
        if not (target / relative).is_file():
            raise FileNotFoundError(f'Output thiếu {target / relative}')
    print('Đã chép:', source, '→', target)
else:
    print('Chạy local: dùng results_224 hiện có trong repository.')


In [ ]:
from pathlib import Path
import os, sys

# Có thể chỉ rõ ba đường dẫn bằng biến môi trường khi gắn repository/dataset trên Kaggle.
root_hint = os.environ.get('CS231_PROJECT_ROOT')
root_candidates = [Path(root_hint)] if root_hint else [Path.cwd(), *Path.cwd().parents]
if Path('/kaggle/input').is_dir():
    root_candidates += list(Path('/kaggle/input').glob('*'))
    root_candidates += list(Path('/kaggle/input').glob('*/*'))
PROJECT_ROOT = next((p.resolve() for p in root_candidates
                     if (p / 'src/SVM/pipeline.py').is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Không tìm thấy repository. Đặt CS231_PROJECT_ROOT.')
sys.path.insert(0, str(PROJECT_ROOT))

data_hint = os.environ.get('CS231_DATA_ROOT')
data_candidates = [Path(data_hint)] if data_hint else [PROJECT_ROOT / 'DATA']
if not data_hint and Path('/kaggle/input').is_dir():
    data_candidates += list(Path('/kaggle/input').glob('*/DATA'))
    data_candidates += list(Path('/kaggle/input').glob('*/*/DATA'))
DATA_ROOT = next((p.resolve() for p in data_candidates
                  if (p / 'train/10000').is_dir()), None)
if DATA_ROOT is None:
    raise FileNotFoundError('Không tìm thấy DATA/train/10000. Đặt CS231_DATA_ROOT.')

result_hint = os.environ.get('CS231_SVM_RESULTS_ROOT')
if result_hint:
    RESULTS_ROOT = Path(result_hint).resolve()
elif Path('/kaggle/working').is_dir():
    RESULTS_ROOT = Path('/kaggle/working/CS231_SVM/results_224')
else:
    RESULTS_ROOT = PROJECT_ROOT / 'src/SVM/results_224'
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
print('Repository:', PROJECT_ROOT)
print('DATA:', DATA_ROOT)
print('Kết quả:', RESULTS_ROOT)


In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from src.SVM.pipeline import (
    CLASSES, FEATURE_FOLDERS, manifest_for_split, extract_matrix, l2,
    save_features, load_features, same_manifest, run_dev_grid, save_dev_result,
    save_json, hsv12, hog_raw,
)
from src.preprocessing import read_rgb224


## 1. Đọc manifest và cấu hình HOG


In [ ]:
train_manifest = pd.read_csv(RESULTS_ROOT / 'inputs/train_manifest.csv')
dev_manifest = pd.read_csv(RESULTS_ROOT / 'inputs/dev_manifest.csv')
same_manifest(train_manifest, manifest_for_split(DATA_ROOT, 'train'))
same_manifest(dev_manifest, manifest_for_split(DATA_ROOT, 'dev'))
OUTPUT = RESULTS_ROOT / 'hog'
OUTPUT.mkdir(parents=True, exist_ok=True)
FEATURE_ID = 'HOG'
FEATURE_PIPELINE = 'RGB224 -> gray -> HOG 9 orientations, cell16, block2, L2-Hys -> whole-vector L2'


## 2. Trực quan hóa ảnh xám và HOG

Hình HOG giúp đọc cấu trúc gradient. SVC nhận descriptor số 6.084 chiều, không nhận hình trực quan.


In [ ]:
import cv2
from skimage.feature import hog
sample = read_rgb224(DATA_ROOT / train_manifest.relative_path.iloc[0])
gray = cv2.cvtColor(sample, cv2.COLOR_RGB2GRAY)
descriptor, visual = hog(gray, orientations=9, pixels_per_cell=(16, 16),
                         cells_per_block=(2, 2), block_norm='L2-Hys',
                         transform_sqrt=False, visualize=True,
                         feature_vector=True, channel_axis=None)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, image, title in zip(axes, [sample, gray, visual],
                            ['RGB224', 'Ảnh xám', 'Trực quan HOG']):
    ax.imshow(image, cmap='gray' if image.ndim == 2 else None)
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.savefig(OUTPUT / 'hog_example.png', dpi=150)
plt.show()
print('Số chiều HOG:', descriptor.size)


## 3. Trích HOG và chuẩn hóa

Vector HOG trước L2 toàn ảnh được lưu riêng để PCA học từ train ở bước 04. SVC nhánh này dùng HOG đã L2 toàn vector.


In [ ]:
X_train_raw = extract_matrix(train_manifest, DATA_ROOT, 'HOG_RAW')
X_dev_raw = extract_matrix(dev_manifest, DATA_ROOT, 'HOG_RAW')
np.save(OUTPUT / 'X_train_hog_before_l2.npy', X_train_raw)
np.save(OUTPUT / 'X_dev_hog_before_l2.npy', X_dev_raw)
X_train, X_dev = l2(X_train_raw), l2(X_dev_raw)
y_train = train_manifest.label_vnd.to_numpy(dtype=np.int64)
y_dev = dev_manifest.label_vnd.to_numpy(dtype=np.int64)
save_features(OUTPUT, train_manifest, dev_manifest, X_train, X_dev)
del X_train_raw, X_dev_raw
print(X_train.shape, X_dev.shape, X_train.dtype)
print('L2 train:', np.linalg.norm(X_train[0]))


## 4. Khảo sát SVC trên dev


In [ ]:
grid, best_pred, best_params = run_dev_grid(X_train, y_train, X_dev, y_dev)
scores = save_dev_result(OUTPUT, FEATURE_ID, grid, best_pred, dev_manifest)
save_json(OUTPUT / 'run_config.json', {
    'feature_id': FEATURE_ID, 'feature_dim': int(X_train.shape[1]),
    'preprocessing': 'RGB uint8 224x224 OpenCV INTER_LINEAR',
    'feature_pipeline': FEATURE_PIPELINE,
    'svm_grid': 'linear C=(0.1,1,10); rbf C=(1,10), gamma=(scale,10)',
    'class_weight': None, 'cache_size_mb': 512,
    'decision_function_shape': 'ovr', 'break_ties': True,
    'best_params': best_params, 'fit_split': 'train', 'selection_split': 'dev',
})
print('Cấu hình thắng trên dev:')
display(grid.head(7))
print('Accuracy:', scores['accuracy'], 'Macro-F1:', scores['macro_f1'],
      'MAE VND/ảnh:', scores['mae_vnd'])
display(pd.DataFrame(scores['per_class'])[
    ['denomination_vnd', 'support', 'precision', 'recall', 'f1']])


## 5. Ma trận nhầm lẫn


In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay(np.asarray(scores['confusion_matrix']),
                       display_labels=CLASSES).plot(ax=ax, cmap='Blues',
                                                     values_format='d', xticks_rotation=45)
ax.set_title(f'{FEATURE_ID} + SVM — dev')
plt.tight_layout()
plt.savefig(OUTPUT / 'confusion_matrix_dev.png', dpi=180)
plt.show()
